# Part 2 — Agentic RAG: ground the agent in a real catalogue

In Part 1 the agent searched products by **keyword**. That works until a
customer does what customers always do — describe what they *want* in their
own words.

Ask Part 1's agent for *"a waterproof jacket for heavy rain"* and it
keyword-matches the word "waterproof". But our best rain jacket, the
Stormshield Hardshell, has a product description that never uses that word
once. Keyword search literally cannot find it.

This part fixes that with **retrieval** — searching the catalogue by
*meaning* instead of by *spelling*. And it makes the retrieval **agentic**:
a tool the agent decides to call, as often as the question needs, rather
than a fixed step bolted on the front.

> **Before you start:** [`../setup.md`](../setup.md) done, and
> `python ../llm.py` prints `connection ok`.

## The one idea

Retrieval-augmented generation (RAG) is three moves:

1. **Embed** every catalogue document into a vector — a list of numbers
   that captures its *meaning*. Do this once, up front.
2. When a question arrives, **embed the question the same way** and find
   the documents whose vectors sit closest to it.
3. **Hand those documents to the model** and ask it to answer using only
   them.

"Closest in meaning" is the trick: *"keeps the rain off"* and *"sheds
sustained downpours"* land near each other in vector space even though they
share no words.

**Agentic** RAG is one step further. Instead of retrieving once before the
model runs, we give the model retrieval as a **tool**. It decides when to
search, what to search for, and whether one search was enough — the exact
model -> tool -> model loop you built in Part 1.

In [ ]:
import sys
from pathlib import Path

sys.path.append("..")  # so we can import llm.py from the repo root
from llm import embed_client, embed_config, embed_extra_body

# The catalogue: one Markdown file per product, in catalog/.
catalog = [
    {"id": path.stem, "text": path.read_text(encoding="utf-8")}
    for path in sorted(Path("catalog").glob("*.md"))
]

EMBED_MODEL = embed_config()["model"]
print(f"Loaded {len(catalog)} product documents. Embedding model: {EMBED_MODEL}")
print()
print(catalog[0]["text"][:260], "...")

## Step 1 — Turn the catalogue into vectors

An **embedding** is a function that maps a piece of text to a fixed-length
list of numbers, built so that texts with similar meaning get similar
numbers. We use NVIDIA's `nv-embedqa-e5-v5` model — 1024 numbers per text.

One subtlety, and it matters: this model is **asymmetric**. It embeds a
*document* and a *search query* differently, and you must tell it which is
which (`input_type` = `"passage"` vs `"query"`). `llm.py` handles that in
`embed_extra_body()` — and returns nothing for providers whose embedding
models do not need the hint, so this notebook stays portable.

We embed all 12 products in **one** API call, then store the vectors in a
[FAISS](https://github.com/facebookresearch/faiss) index — a small library
that does one job well: given a vector, find the nearest stored vectors.

In [ ]:
import faiss
import numpy as np

# Embed all 12 product documents in ONE batched call. input_type="passage"
# tells NVIDIA's model these are documents to be searched, not queries.
response = embed_client().embeddings.create(
    model=EMBED_MODEL,
    input=[doc["text"] for doc in catalog],
    extra_body=embed_extra_body("passage"),
)

# Keep the vector order aligned with `catalog`.
ordered = sorted(response.data, key=lambda item: item.index)
vectors = np.array([item.embedding for item in ordered], dtype="float32")
faiss.normalize_L2(vectors)  # normalise so inner product == cosine similarity

index = faiss.IndexFlatIP(vectors.shape[1])
index.add(vectors)
print(f"Indexed {index.ntotal} products, {vectors.shape[1]} numbers per vector.")

## Step 2 — Search by meaning

`retrieve()` is the whole search engine: embed the question (this time as a
`"query"`), ask FAISS for the nearest product vectors, return them. The
score is cosine similarity — 1.0 is identical meaning, 0.0 is unrelated.

In [ ]:
def retrieve(query: str, k: int = 3):
    """Return the k catalogue documents closest in meaning to `query`."""
    response = embed_client().embeddings.create(
        model=EMBED_MODEL,
        input=[query],
        extra_body=embed_extra_body("query"),  # a query, not a passage
    )
    q = np.array([response.data[0].embedding], dtype="float32")
    faiss.normalize_L2(q)

    scores, ids = index.search(q, k)
    return [
        {"id": catalog[int(i)]["id"], "score": float(s), "text": catalog[int(i)]["text"]}
        for s, i in zip(scores[0], ids[0])
    ]


print('retrieve("a waterproof jacket for heavy rain"):')
for hit in retrieve("a waterproof jacket for heavy rain"):
    print(f'  {hit["score"]:.3f}  {hit["id"]}')

## Why this beats keyword search

That top hit — **JKT-001**, the Stormshield Hardshell — is found even
though its product description never contains the word "waterproof". See
for yourself what a plain keyword search does with the same need:

In [ ]:
# A naive keyword search: which documents literally contain the word?
keyword_hits = [doc["id"] for doc in catalog if "waterproof" in doc["text"].lower()]
print('Keyword search for "waterproof" :', keyword_hits)

semantic_hits = [hit["id"] for hit in retrieve("a waterproof jacket for heavy rain")]
print("Semantic search, same need     :", semantic_hits)

Keyword search finds only the **boot** (FTW-001 — its description happens
to use the word "waterproof") and never reaches the **jacket** the customer
asked for. Semantic search ranks JKT-001 first, because *"waterproof jacket
for heavy rain"* and *"sheds sustained downpours ... shrugs off driving
rain"* mean the same thing. That gap is the whole reason to embed a
catalogue.

## Step 3 — Plain RAG: retrieve, then answer

The simplest way to use retrieval: fetch the relevant documents, paste them
into the prompt, and ask the model to answer from them. One retrieval, one
answer.

In [ ]:
from llm import EXTRA_BODY, llm_config, raw_client

CHAT_MODEL = llm_config()["model"]


def plain_rag(question: str) -> str:
    hits = retrieve(question, k=3)
    context = "\n\n---\n\n".join(hit["text"] for hit in hits)
    prompt = (
        "You are a shopping assistant for an outdoor-gear shop. Answer the "
        "customer using ONLY the product documents below. Cite every product "
        "you mention by its id in square brackets, e.g. [JKT-001]. If the "
        "documents do not answer the question, say so.\n\n"
        f"PRODUCT DOCUMENTS:\n{context}\n\n"
        f"CUSTOMER QUESTION: {question}"
    )
    reply = raw_client().chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1,
        extra_body=EXTRA_BODY,
    )
    return reply.choices[0].message.content or "(empty response)"


print(plain_rag("What should I wear to stay dry in a downpour?"))

## Step 4 — Agentic RAG: retrieval as a tool

Plain RAG retrieves **once**, with the customer's question used verbatim as
the search, then answers. It has no memory and cannot handle a follow-up.
Real shoppers do not work that way — they ask, then ask again.

We wrap `retrieve()` as a `@tool` and hand it to `create_agent` — the same
LangGraph agent from Part 1. Retrieval becomes something the model
*chooses* to do: it reformulates the request into good search terms,
decides when a search is needed, and — with a checkpointer for memory —
carries context from one turn to the next.

Watch the two-turn conversation below. The second question only says "it",
and the agent still knows what "it" refers to.

In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver

from llm import chat_model


@tool
def search_catalog(query: str) -> str:
    """Search the outdoor-gear catalogue by meaning and return the most
    relevant product documents. Pass concise search terms, not a whole
    sentence."""
    hits = retrieve(query, k=3)
    print(f'  [search_catalog] "{query}" -> {[hit["id"] for hit in hits]}')
    return "\n\n---\n\n".join(hit["text"] for hit in hits)


SYSTEM_PROMPT = """You are a shopping assistant for an outdoor-gear shop.

- Use search_catalog to find products before you answer. Reformulate the
  customer's request into short search terms.
- Answer only from what the tool returns. Never invent products, prices or
  stock, and never present a product as something it is not. If an item is
  out of stock, say so plainly.
- Cite every product you mention by its id in square brackets, e.g. [JKT-001].
- Recommend at most two products and keep the answer short."""

rag_agent = create_agent(
    model=chat_model(temperature=0.1),
    tools=[search_catalog],
    system_prompt=SYSTEM_PROMPT,
    checkpointer=InMemorySaver(),
)
conversation = {"configurable": {"thread_id": "shopper-1"}}

q1 = "I need a warm jacket for cold, dry winter days."
first = rag_agent.invoke({"messages": [{"role": "user", "content": q1}]}, conversation)
print("\nQ1:", q1)
print("A1:", first["messages"][-1].content)

q2 = "Which small daypack could I carry it in?"
second = rag_agent.invoke({"messages": [{"role": "user", "content": q2}]}, conversation)
print("\nQ2:", q2)
print("A2:", second["messages"][-1].content)

## Answers you can trace — and what is still missing

Look back at the run. The agent searched the catalogue once per question
(the `[search_catalog]` lines), turning each request into short search
terms, and it answered the follow-up — "carry **it**" — from memory of the
first turn. Every product is cited by id, so every claim traces back to a
retrieved document. That traceability is the real product of RAG.

But grounded is not the same as correct:

- **Retrieval can fetch the wrong document** — and the agent will answer
  confidently from it anyway.
- **The agent can still misread** a document it retrieved: quote the wrong
  price, miss an "out of stock".
- **Twelve Markdown files is not a catalogue.** Ten thousand SKUs need
  chunking, metadata filters (price, stock, category) and an index that
  stays fresh as inventory changes.

Nothing here *tells you* when the agent is wrong — it always produces a
fluent answer.

---

**From demo to production.** Before a retrieval agent meets real customers,
one question has to be answered: *how often is it right, and how do you
know?* **Part 3** builds the evaluation and guardrail harness that measures
exactly that — the line between a demo you can show and an agent you can
trust with revenue.